# Banking ETL Demo — PySpark + Delta Lake
## Start here
Import this `.ipynb` into your Databricks Workspace, attach Serverless or compatible compute, and run all cells in order.
No uploads, external connections, packages, secrets, DBFS paths, or RDDs are needed.

**Pipeline:** embedded fictional CSV records → Bronze raw Delta tables → Silver validated Delta tables → Gold account summary → automated checks.
The embedded CSV records stand in for an external source. This is a batch learning demo, not a live bank integration.

**Configuration:** the next cell uses your current catalog and creates a dedicated `banking_etl_demo` schema. You need permission to create schemas/tables; otherwise set SCHEMA to a schema you can write to.
Each run replaces only the project tables named in this notebook. Use a dedicated schema. There are no DROP commands.

**Business rules:** Credit increases the balance; Debit decreases it. For this demo, the balances from your original notebook are explicitly treated as **opening balances**. All money is INR using decimal(18,2). Current inactive status is flagged for review, not used to reject historical transactions because status history is unavailable.


In [ ]:
from pyspark.sql import functions as F

CATALOG = spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = "banking_etl_demo"  # Change if your workspace requires an existing schema.

def quoted(identifier):
    return "`" + identifier.replace("`", "``") + "`"

namespace = f"{quoted(CATALOG)}.{quoted(SCHEMA)}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {namespace}")

def table_name(name):
    return f"{namespace}.{quoted(name)}"

def save_delta(df, name):
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").saveAsTable(table_name(name)))

def read_table(name):
    return spark.table(table_name(name))

print("Project namespace:", namespace)


## 1. Extract sample CSV records
The original 5 customers, 5 accounts and 8 transactions are retained. Transactions also contain 1 exact duplicate and 5 invalid rows to demonstrate testing.
For simplicity, these embedded records contain no quoted commas. `from_csv` parses each record using an explicit string schema, preserving raw values in Bronze.


In [ ]:
customers_csv = """101,Amit,Lucknow,2023-01-15
102,Priya,Delhi,2023-03-20
103,Rahul,Noida,2023-06-10
104,Neha,Kanpur,2024-02-05
105,Vikas,Varanasi,2024-05-12"""
accounts_csv = """1001,101,Savings,25000.00,Active
1002,102,Current,85000.00,Active
1003,103,Savings,12000.00,Active
1004,104,Savings,45000.00,Inactive
1005,105,Current,95000.00,Active"""
transactions_csv = """5001,1001,2026-10-01 10:15:00,Credit,5000.00,UPI
5002,1001,2026-10-01 11:30:00,Debit,1200.00,ATM
5003,1002,2026-10-02 09:00:00,Debit,15000.00,NEFT
5004,1003,2026-10-02 14:45:00,Credit,8000.00,UPI
5005,1004,2026-10-03 16:20:00,Debit,2000.00,ATM
5006,1005,2026-10-04 12:10:00,Debit,50000.00,NEFT
5007,1002,2026-10-04 13:30:00,Credit,25000.00,NEFT
5008,1001,2026-10-05 17:00:00,Debit,700.00,UPI
5001,1001,2026-10-01 10:15:00,Credit,5000.00,UPI
5009,9999,2026-10-06 10:00:00,Credit,100.00,UPI
5010,1001,2026-10-06 11:00:00,Debit,-50.00,ATM
5011,1002,2026-10-06 12:00:00,Credit,,NEFT
5012,1003,not-a-date,Credit,100.00,UPI
5013,1005,2026-10-06 14:00:00,Transfer,100.00,UPI"""

customer_columns = ["customer_id", "customer_name", "city", "join_date"]
account_columns = ["account_id", "customer_id", "account_type", "balance", "account_status"]
transaction_columns = ["transaction_id", "account_id", "transaction_timestamp",
                       "transaction_type", "amount", "channel"]

def parse_csv_records(text, columns):
    lines = [(line,) for line in text.strip().splitlines()]
    schema = ", ".join(f"{c} STRING" for c in columns)
    return (spark.createDataFrame(lines, "value STRING")
            .select(F.from_csv("value", schema, {"mode": "FAILFAST"}).alias("row"))
            .select("row.*"))

customers_raw = parse_csv_records(customers_csv, customer_columns)
accounts_raw = parse_csv_records(accounts_csv, account_columns)
transactions_raw = parse_csv_records(transactions_csv, transaction_columns)


## 2. Load Bronze — preserve raw source values
Ingestion time is metadata; business columns remain strings. Exact duplicates remain visible here.

In [ ]:
for df, name in [(customers_raw, "bronze_customers"),
                 (accounts_raw, "bronze_accounts"),
                 (transactions_raw, "bronze_transactions")]:
    save_delta(df.withColumn("ingested_at", F.current_timestamp()), name)

display(read_table("bronze_transactions").orderBy("transaction_id"))


## 3. Transform customer and account master data
Cast identifiers and dates, trim text, and use decimal money. For this fixed demo, invalid master data fails the pipeline before transaction processing.

In [ ]:
customers_df = read_table("bronze_customers").selectExpr(
    "try_cast(customer_id AS BIGINT) AS customer_id",
    "trim(customer_name) AS customer_name", "trim(city) AS city",
    "try_cast(join_date AS DATE) AS join_date")
accounts_df = read_table("bronze_accounts").selectExpr(
    "try_cast(account_id AS BIGINT) AS account_id",
    "try_cast(customer_id AS BIGINT) AS customer_id",
    "trim(account_type) AS account_type",
    "try_cast(balance AS DECIMAL(18,2)) AS opening_balance",
    "upper(trim(account_status)) AS account_status")

assert customers_df.filter("customer_id IS NULL OR customer_name IS NULL OR customer_name = '' OR join_date IS NULL").count() == 0, "Invalid customer master"
assert customers_df.count() == customers_df.select("customer_id").distinct().count(), "Duplicate customer IDs"
assert accounts_df.filter("account_id IS NULL OR customer_id IS NULL OR opening_balance IS NULL OR opening_balance < 0 OR account_status IS NULL OR account_status NOT IN ('ACTIVE', 'INACTIVE')").count() == 0, "Invalid account master"
assert accounts_df.count() == accounts_df.select("account_id").distinct().count(), "Duplicate account IDs"
assert accounts_df.join(customers_df, "customer_id", "left_anti").count() == 0, "Unknown customer"
save_delta(customers_df, "silver_customers")
save_delta(accounts_df, "silver_accounts")


## 4. Validate transactions and separate rejections
Remove only exact duplicates across all source business columns. Conflicting rows with the same transaction ID are all rejected, avoiding arbitrary selection.
Reject missing/invalid IDs, unknown accounts, bad timestamps, unsupported transaction types, and missing/nonpositive amounts. A rejection may have multiple reasons.


In [ ]:
raw_transactions = read_table("bronze_transactions").select(*transaction_columns)
deduplicated = raw_transactions.dropDuplicates(transaction_columns)
duplicate_rows_removed = raw_transactions.count() - deduplicated.count()
parsed = deduplicated.selectExpr(
    "try_cast(transaction_id AS BIGINT) AS transaction_id",
    "try_cast(account_id AS BIGINT) AS account_id",
    "try_cast(transaction_timestamp AS TIMESTAMP) AS transaction_timestamp",
    "upper(trim(transaction_type)) AS transaction_type",
    "try_cast(amount AS DECIMAL(18,2)) AS amount", "upper(trim(channel)) AS channel",
    "transaction_id AS raw_transaction_id", "account_id AS raw_account_id",
    "transaction_timestamp AS raw_transaction_timestamp", "amount AS raw_amount",
    "transaction_type AS raw_transaction_type", "channel AS raw_channel")
conflicts = (parsed.filter("transaction_id IS NOT NULL").groupBy("transaction_id")
             .count().filter("count > 1").select("transaction_id")
             .withColumn("conflicting_id", F.lit(True)))
known_accounts = accounts_df.select(F.col("account_id").alias("known_account_id"))
checked = (parsed.join(conflicts, "transaction_id", "left")
           .join(known_accounts, parsed.account_id == known_accounts.known_account_id, "left"))
checked = checked.withColumn("rejection_reason", F.concat_ws("; ",
    F.when(F.col("transaction_id").isNull(), "Invalid transaction ID"),
    F.when(F.col("conflicting_id").isNotNull(), "Conflicting transaction ID"),
    F.when(F.col("account_id").isNull(), "Invalid account ID"),
    F.when(F.col("account_id").isNotNull() & F.col("known_account_id").isNull(), "Unknown account"),
    F.when(F.col("transaction_timestamp").isNull(), "Invalid timestamp"),
    F.when(F.col("transaction_type").isNull() | ~F.col("transaction_type").isin("CREDIT", "DEBIT"), "Invalid transaction type"),
    F.when(F.col("amount").isNull() | (F.col("amount") <= 0), "Invalid amount")
))
transactions_df = checked.filter("rejection_reason = ''").select(*transaction_columns)
rejected_transactions_df = checked.filter("rejection_reason <> ''").drop("known_account_id", "conflicting_id")
save_delta(transactions_df, "silver_transactions")
save_delta(rejected_transactions_df, "rejected_transactions")
display(rejected_transactions_df.orderBy("transaction_id"))


## 5. Create Gold — account-level report
Start from all accounts so accounts without transactions still appear. Flag current inactive status for review. Calculated balances use the stated opening-balance assumption.

In [ ]:
transactions_df = read_table("silver_transactions")
zero = F.lit("0.00").cast("decimal(18,2)")
totals = transactions_df.groupBy("account_id").agg(
    F.sum(F.when(F.col("transaction_type") == "CREDIT", F.col("amount")).otherwise(zero)).alias("total_credits"),
    F.sum(F.when(F.col("transaction_type") == "DEBIT", F.col("amount")).otherwise(zero)).alias("total_debits"),
    F.count("*").alias("transaction_count"))
account_summary_df = (read_table("silver_accounts")
    .join(read_table("silver_customers"), "customer_id", "left")
    .join(totals, "account_id", "left")
    .fillna({"total_credits": 0, "total_debits": 0, "transaction_count": 0})
    .withColumn("calculated_balance", F.col("opening_balance") + F.col("total_credits") - F.col("total_debits"))
    .withColumn("review_required", (F.col("account_status") == "INACTIVE") & (F.col("transaction_count") > 0))
    .select("account_id", "customer_id", "customer_name", "city", "account_type", "account_status",
            "opening_balance", "total_credits", "total_debits", "transaction_count", "calculated_balance", "review_required"))
save_delta(account_summary_df, "gold_account_summary")
display(read_table("gold_account_summary").orderBy("account_id"))


## 6. Automated ETL checks
These expectations are specific to the supplied fixtures. If you change sample data, update the expected values. Failures stop the notebook with an explanatory assertion.

In [ ]:
from decimal import Decimal

results = []
def check(name, actual, expected):
    passed = actual == expected
    results.append((name, str(expected), str(actual), "PASS" if passed else "FAIL"))

bronze_count = read_table("bronze_transactions").count()
silver = read_table("silver_transactions")
rejected = read_table("rejected_transactions")
gold = read_table("gold_account_summary")
check("Bronze transaction rows", bronze_count, 14)
check("Exact duplicate rows removed", duplicate_rows_removed, 1)
check("Rejected transaction rows", rejected.count(), 5)
check("Silver transaction rows", silver.count(), 8)
check("Row reconciliation", silver.count() + rejected.count() + duplicate_rows_removed, bronze_count)
check("Unique Silver transaction IDs", silver.select("transaction_id").distinct().count(), silver.count())
check("Silver account references", silver.join(read_table("silver_accounts"), "account_id", "left_anti").count(), 0)
check("Gold account rows", gold.count(), 5)
check("Gold transaction count", gold.agg(F.sum("transaction_count")).first()[0], 8)
check("Gold credit total", gold.agg(F.sum("total_credits")).first()[0], Decimal("38000.00"))
check("Gold debit total", gold.agg(F.sum("total_debits")).first()[0], Decimal("68900.00"))
check("Gold balance total", gold.agg(F.sum("calculated_balance")).first()[0], Decimal("231100.00"))
expected = {1001: ("5000", "1900", 3, "28100"), 1002: ("25000", "15000", 2, "95000"),
            1003: ("8000", "0", 1, "20000"), 1004: ("0", "2000", 1, "43000"),
            1005: ("0", "50000", 1, "45000")}
for row in gold.collect():
    credits, debits, count, balance = expected[row.account_id]
    check(f"Account {row.account_id} summary",
          (row.total_credits, row.total_debits, row.transaction_count, row.calculated_balance),
          (Decimal(credits), Decimal(debits), count, Decimal(balance)))
check("Inactive accounts flagged", [r.account_id for r in gold.filter("review_required").orderBy("account_id").collect()], [1004])
check("Rejected transaction IDs", [r.transaction_id for r in rejected.orderBy("transaction_id").collect()], [5009, 5010, 5011, 5012, 5013])
validation_results_df = spark.createDataFrame(results, "check_name STRING, expected STRING, actual STRING, status STRING")
save_delta(validation_results_df, "etl_test_results")
display(validation_results_df)
assert all(r[3] == "PASS" for r in results), "ETL checks failed: inspect etl_test_results"
print("All ETL checks passed.")


## 7. Query the saved report
The tables persist after the notebook session. Copy the printed fully qualified table name into SQL Editor if needed.

In [ ]:
print("Report table:", table_name("gold_account_summary"))
display(spark.sql(f"""
SELECT account_id, customer_name, total_credits, total_debits,
       transaction_count, calculated_balance, review_required
FROM {table_name('gold_account_summary')}
ORDER BY account_id
"""))


## Demo talk track
1. Show the raw 14 transactions in Bronze, including bad data and the duplicate.
2. Explain exact deduplication and show the 5 rejected transactions with reasons.
3. Show the 8 valid transactions in Silver and decimal monetary types.
4. Show Gold: 5 account summaries, with account 1004 flagged for review.
5. Show the test results and explain row reconciliation: **14 = 8 valid + 5 rejected + 1 duplicate**.
6. Rerun all cells: the demo overwrites its own tables, so counts do not accumulate.

## Tables created
- bronze_customers, bronze_accounts, bronze_transactions
- silver_customers, silver_accounts, silver_transactions
- rejected_transactions
- gold_account_summary
- etl_test_results

## Troubleshooting and scope
- Permission error creating schema: choose an existing writable catalog/schema in configuration.
- Table write error: check CREATE TABLE and MODIFY privileges and compute availability.
- Run cells in order or use Run all; later cells depend on earlier definitions.
- The notebook replaces these named demo tables; do not point it at production tables.
- This demo is a full-refresh batch. It does not implement incremental ingestion, account status history, overdraft checks or production orchestration.
- To demonstrate a zero-transaction account, add an account with no transactions and update the fixture-specific assertions.

## Documentation
- [Import notebooks](https://docs.databricks.com/aws/en/notebooks/notebook-export-import)
- [Create and manage Delta tables](https://docs.databricks.com/aws/en/delta/tutorial)
- [CREATE SCHEMA](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-ddl-create-schema)

## Verification before delivery
Notebook JSON and Python cell syntax were checked locally. Sample counts and account totals were independently calculated. Databricks execution must be performed in your workspace; no connection to your compute was available during creation.
